## Imports and Configuration

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import librosa
from tqdm.notebook import tqdm
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
DATASET_PATH = r"Dataset"

SAMPLE_RATE = 22050
DURATION = 4
SAMPLES_PER_TRACK = SAMPLE_RATE * DURATION

N_MFCC = 20
HOP_LENGTH = 512
N_FFT = 2048

LANGUAGES = ['German', 'Spanish', 'Italian', 'Korean']
GENDERS = ['Male', 'Female']
PREPROCESSES_FILE = "processed_features_dataset.csv"

## Extracting Features

In [3]:
def extract_features(y, sr):
    """
    Extracts comprehensive audio features from a given audio signal.

    Args:
        y (np.ndarray): The audio time-series array.
        sr (int): The sampling rate of the audio.

    Returns:
        dict: A dictionary mapping feature names (str) to their computed values (float).
              Example keys: 'mfcc_1_mean', 'mfcc_1_var', 'centroid_mean', ...
    """
    features = {}
    
    # 1. MFCC (Timbre / Phonemes)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    for i in range(N_MFCC):
        features[f'mfcc_{i+1}_mean'] = np.mean(mfcc[i])
        features[f'mfcc_{i+1}_var'] = np.var(mfcc[i])

    # 2. Spectral Centroid (Brightness)
    cent = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    features['centroid_mean'] = np.mean(cent)
    features['centroid_var'] = np.var(cent)

    # 3. Zero Crossing Rate (Noisiness / Percussive sounds)
    zcr = librosa.feature.zero_crossing_rate(y)
    features['zcr_mean'] = np.mean(zcr)
    features['zcr_var'] = np.var(zcr)

    # 4. Chroma (Pitch / Tonal content)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    features['chroma_mean'] = np.mean(chroma)
    features['chroma_var'] = np.var(chroma)

    # 5. Spectral Contrast (Peaks vs Valleys in spectrum)
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    features['contrast_mean'] = np.mean(contrast)
    features['contrast_var'] = np.var(contrast)

    return features

## Processing Files

In [4]:
def process_file(file_path, label, gender, filename):
    """
    Processes a single audio file by loading, cleaning, segmenting, and extracting features.

    Args:
        file_path (str): Full path to the audio file.
        label (str): The language label (e.g., 'German', 'Spanish').
        gender (str): The gender of the speaker ('Male' or 'Female').
        filename (str): The original name of the file (for tracking purposes).

    Returns:
        list: A list of dictionaries, where each dictionary represents 
              the features and metadata for a single 4-second segment.
    """
    data_points = []
    
    try:
        # 1. Load File
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
        
        # 2. Trim Silence (Remove silence from start and end with threshold)
        y_trimmed, _ = librosa.effects.trim(y, top_db=20)
        
        # 3. Normalize (Scale amplitude between -1 and 1)
        y_norm = librosa.util.normalize(y_trimmed)
        
        # 4. Calculate number of segments
        total_length = len(y_norm)
        num_segments = int(total_length / SAMPLES_PER_TRACK)
        
        # 5. Segmentation Loop
        for i in range(num_segments):
            start = i * SAMPLES_PER_TRACK
            finish = start + SAMPLES_PER_TRACK
            
            segment = y_norm[start:finish]
            
            if len(segment) == SAMPLES_PER_TRACK:
                # Extract Features
                features = extract_features(segment, sr)
                
                # Add Metadata
                features['label'] = label       # Target Label (Language)
                features['gender'] = gender     # Speaker Gender
                features['filename'] = filename # Source Filename
                
                data_points.append(features)
                
    except Exception as e:
        print(f"Error processing {filename}: {e}")
        
    return data_points

## Execution Part

In [5]:
final_dataset = []

print("Starting Feature Extraction Process...")

total_files = sum([len(glob.glob(os.path.join(DATASET_PATH, lang, gen, "*"))) for lang in LANGUAGES for gen in GENDERS])

with tqdm(total=total_files, desc="Processing Files") as pbar:
    for lang in LANGUAGES:
        for gender in GENDERS:
            folder_path = os.path.join(DATASET_PATH, lang, gender)
            
            files = glob.glob(os.path.join(folder_path, "*.mp3"))
            
            for file_path in files:
                filename = os.path.basename(file_path)
                
                segments_features = process_file(file_path, lang, gender, filename)
                
                final_dataset.extend(segments_features)
                
                pbar.update(1)

print("Processing Complete!")
print(f"Total Segments Created: {len(final_dataset)}")

Starting Feature Extraction Process...


Processing Files:   0%|          | 0/720 [00:00<?, ?it/s]

Processing Complete!
Total Segments Created: 10852


In [6]:
df_preprocesses = pd.DataFrame(final_dataset)

print("Data Shape:", df_preprocesses.shape)
display(df_preprocesses.head())

output_filename = PREPROCESSES_FILE
df_preprocesses.to_csv(output_filename, index=False)

print(f"Dataset saved to: {output_filename}")

Data Shape: (10852, 51)


,mfcc_1_mean,mfcc_1_var,mfcc_2_mean,mfcc_2_var,mfcc_3_mean,mfcc_3_var,mfcc_4_mean,mfcc_4_var,mfcc_5_mean,mfcc_5_var,...,centroid_var,zcr_mean,zcr_var,chroma_mean,chroma_var,contrast_mean,contrast_var,label,gender,filename
0,-263.922028,18606.697266,77.982285,3757.337402,-9.310464,984.790100,39.328022,945.080322,5.222437,355.088684,...,1.277559e+06,0.127055,0.007808,0.395074,0.095600,23.148505,163.087572,German,Male,810100147_male_german_voice02.mp3
1,-349.743774,26918.447266,63.186958,3615.752197,-9.667653,981.036682,50.941124,904.139099,11.845339,385.063263,...,1.027629e+06,0.124588,0.008629,0.368092,0.102901,24.040477,160.237843,German,Male,810100147_male_german_voice02.mp3
2,-245.285095,19061.607422,73.344009,3277.503662,-8.776618,1122.651611,47.934101,1196.814819,7.272524,265.862091,...,1.245263e+06,0.114661,0.009235,0.352566,0.099449,23.742645,158.019540,German,Male,810100147_male_german_voice02.mp3
3,-318.250305,27300.646484,68.638039,4238.564941,3.514310,799.790710,29.387894,834.465698,-0.657580,582.343628,...,1.787097e+06,0.127870,0.009768,0.412828,0.093851,23.488550,153.645357,German,Male,810100147_male_german_voice02.mp3
4,-339.062805,33127.367188,71.513077,4391.493652,-1.589328,1419.516235,25.225393,737.102417,0.498458,531.279419,...,1.657907e+06,0.120823,0.009985,0.391374,0.098067,24.110493,149.460309,German,Male,810100147_male_german_voice02.mp3


Dataset saved to: processed_features_dataset.csv


In [7]:
df = pd.read_csv(output_filename)

print(f"File '{output_filename}' loaded successfully!")
print(f"Total Samples (Segments): {len(df)}")
print(f"Number of Features (Columns): {df.shape[1]}")

print(df['label'].value_counts())

null_counts = df.isnull().sum().sum()
print(f"Total Null/Missing Values: {null_counts}")

display(df.tail())

File 'processed_features_dataset.csv' loaded successfully!
Total Samples (Segments): 10852
Number of Features (Columns): 51
label
Italian    2954
German     2675
Korean     2618
Spanish    2605
Name: count, dtype: int64
Total Null/Missing Values: 0


,mfcc_1_mean,mfcc_1_var,mfcc_2_mean,mfcc_2_var,mfcc_3_mean,mfcc_3_var,mfcc_4_mean,mfcc_4_var,mfcc_5_mean,mfcc_5_var,...,centroid_var,zcr_mean,zcr_var,chroma_mean,chroma_var,contrast_mean,contrast_var,label,gender,filename
10847,-481.38150,15704.051,17.712465,1784.7109,3.976513,166.36055,4.340812,191.77121,5.175166,220.85180,...,1.911547e+06,0.026144,0.003605,0.197631,0.109437,13.627427,49.141113,Korean,Female,810104272_female_korean_voice18.mp3
10848,-443.80344,28727.723,32.832000,2531.8438,1.654820,366.29050,3.713779,227.47133,11.419375,405.59283,...,2.831062e+06,0.069641,0.012700,0.300700,0.127709,16.275933,90.190102,Korean,Female,810104272_female_korean_voice18.mp3
10849,-312.16226,25044.855,59.023580,3512.0376,16.003382,1430.79750,21.538147,777.66925,27.462204,530.56335,...,2.294154e+06,0.068303,0.007163,0.282150,0.109513,20.651994,132.438731,Korean,Female,810104272_female_korean_voice18.mp3
10850,-211.78299,9454.007,81.633170,3691.4087,18.326185,2228.17140,17.967945,828.17560,37.249763,365.50540,...,2.683024e+06,0.112257,0.017349,0.346296,0.103591,23.083488,145.769853,Korean,Female,810104272_female_korean_voice18.mp3
10851,-277.16486,22022.615,64.456410,4029.7761,18.476974,1530.73050,19.780880,1191.27560,25.707521,415.16370,...,2.605146e+06,0.120532,0.018617,0.360665,0.106926,22.149059,152.960690,Korean,Female,810104272_female_korean_voice18.mp3
